# Imports

In [ ]:
import sys
import os
import numpy as np

sys.path.append(os.path.abspath('../src'))
from D_Compute_Spectrograms.load_eeg_epochs import load_eeg_epochs
from C_EEG_Epoching.load_clean_eeg import load_clean_eeg
from D_Compute_Spectrograms.load_interpolation_config import load_interpolation_config
from D_Compute_Spectrograms.interpolate_epoch_groups import interpolate_epoch_groups
from D_Compute_Spectrograms.create_epochs_from_interpolated_data import create_epochs_from_interpolated_data
from D_Compute_Spectrograms.compute_tfr_by_event import compute_tfr_by_event
from D_Compute_Spectrograms.save_tfr_results import save_tfr_results

# Variables

In [ ]:
subjectID = 'Pilote001'
date      = '2026-07-09'
task      = 'Task1_PPS'
data_path =  '/Users/floremunier/Library/CloudStorage/Dropbox-NeuroRestore/Flore Munier-Jolain/PercepPD/Data'
file_name = f"{subjectID}_{date}_{task}"

baseline_time_start = -0.5
baseline_time_stop = 0.0
N_baseline_points = 500

### Load data

In [3]:
file_path = os.path.join(data_path, subjectID, task,'Output','epoched_EEG')
file_name = f"{subjectID}_{date}_{task}"
epochs, events_id, events_detailed = load_eeg_epochs(file_path, file_name)

raw_cleaned, sfreq, descriptions = load_clean_eeg(data_path, subjectID, date, task)

config_path_raw = os.path.join(os.path.dirname(data_path), 'Code', 'EEG_analysis', 'Scripts', 'src', 'D_Compute_Spectrograms', 'interpolation_config_per_trial_type.json')
interpolation_config_raw = load_interpolation_config(config_path_raw)

config_path_marged = os.path.join(os.path.dirname(data_path), 'Code', 'EEG_analysis', 'Scripts', 'src', 'D_Compute_Spectrograms', 'interpolation_config_for_merging.json')
interpolation_config_merged = load_interpolation_config(config_path_marged)

[load_eeg_epochs] Loading epochs...
  → /Users/floremunier/Library/CloudStorage/Dropbox-NeuroRestore/Flore Munier-Jolain/PercepPD/Data/Pilote001/Task1_PPS/Output/epoched_EEG/Pilote001_2026-07-09_Task1_PPS_epochs.fif


/Users/floremunier/Library/CloudStorage/Dropbox-NeuroRestore/Flore Munier-Jolain/PercepPD/Code/EEG_analysis/Scripts/src/D_Compute_Spectrograms/load_eeg_epochs.py:60: RuntimeWarning: This filename (/Users/floremunier/Library/CloudStorage/Dropbox-NeuroRestore/Flore Munier-Jolain/PercepPD/Data/Pilote001/Task1_PPS/Output/epoched_EEG/Pilote001_2026-07-09_Task1_PPS_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(


[load_eeg_epochs] Loading event ID...
  → /Users/floremunier/Library/CloudStorage/Dropbox-NeuroRestore/Flore Munier-Jolain/PercepPD/Data/Pilote001/Task1_PPS/Output/epoched_EEG/Pilote001_2026-07-09_Task1_PPS_event_id.json
[load_eeg_epochs] Loading events metadata...
  → /Users/floremunier/Library/CloudStorage/Dropbox-NeuroRestore/Flore Munier-Jolain/PercepPD/Data/Pilote001/Task1_PPS/Output/epoched_EEG/Pilote001_2026-07-09_Task1_PPS_events_detailed.json
[load_eeg_epochs] Done.
    Number of epochs: 389
    Event types: 33
    Metadata entries: 432
[load_clean_eeg] Loading cleaned EEG file:
  → /Users/floremunier/Library/CloudStorage/Dropbox-NeuroRestore/Flore Munier-Jolain/PercepPD/Data/Pilote001/Task1_PPS/Output/clean_EEG/Pilote001_2026-07-09_Task1_PPS_clean_raw.fif
[load_interpolation_config] Interpolation config loaded
  → /Users/floremunier/Library/CloudStorage/Dropbox-NeuroRestore/Flore Munier-Jolain/PercepPD/Code/EEG_analysis/Scripts/src/D_Compute_Spectrograms/interpolation_config_

### Interpolate raw Epochs

In [4]:
interp_data_raw, _ = interpolate_epoch_groups(
    epochs,
    events_detailed,
    raw_cleaned,
    interpolation_config_raw
)

sfreq = raw_cleaned.info["sfreq"]
interpolated_epochs_raw, _ = create_epochs_from_interpolated_data(interp_data_raw, sfreq, N_baseline_points, baseline_time_start, ch_names=epochs.ch_names)

--------------------------------
Interpolated Epochs created
--------------------------------
Number of epochs : 389
Number of channels : 63
Number of points : 2600
Sampling frequency : 1000.0 Hz
tmin : -0.5000 s
t=0 located at sample 500 (expected 499)

Event IDs:
  1: VibrotactileOnly_D4_Narrow_Slow
  2: VibrotactileOnly_D3_Narrow_Fast
  3: VisualOnly_Narrow_Slow
  4: VisualOnly_Narrow_Fast
  5: Both_D4_Narrow_Slow
  6: Both_D3_Narrow_Fast
  7: Both_D5_Narrow_Slow
  8: Both_D5_Narrow_Fast
  9: Both_D6_Narrow_Fast
  10: VibrotactileOnly_D7_Narrow_Slow
  11: Both_D6_Narrow_Slow
  12: VibrotactileOnly_D1_Narrow_Fast
  13: Both_D1_Narrow_Slow
  14: Both_D2_Narrow_Slow
  15: VibrotactileOnly_D1_Narrow_Slow
  16: Both_D7_Narrow_Fast
  17: VibrotactileOnly_D7_Narrow_Fast
  18: VibrotactileOnly_D2_Narrow_Slow
  19: VibrotactileOnly_D4_Narrow_Fast
  20: VibrotactileOnly_D5_Narrow_Slow
  21: VibrotactileOnly_D6_Narrow_Fast
  22: Both_D7_Narrow_Slow
  23: VibrotactileOnly_D6_Narrow_Slow
  24: V

### Interpolate for merging

In [5]:
interp_data_merged, _ = interpolate_epoch_groups(
    epochs,
    events_detailed,
    raw_cleaned,
    interpolation_config_merged
)

sfreq = raw_cleaned.info["sfreq"]
interpolated_epochs_merged, _ = create_epochs_from_interpolated_data(interp_data_merged, sfreq, N_baseline_points, baseline_time_start, ch_names=epochs.ch_names)

--------------------------------
Interpolated Epochs created
--------------------------------
Number of epochs : 389
Number of channels : 63
Number of points : 2600
Sampling frequency : 1000.0 Hz
tmin : -0.5000 s
t=0 located at sample 500 (expected 499)

Event IDs:
  1: VibrotactileOnly_D4_Narrow_Slow
  2: VibrotactileOnly_D3_Narrow_Fast
  3: VisualOnly_Narrow_Slow
  4: VisualOnly_Narrow_Fast
  5: Both_D4_Narrow_Slow
  6: Both_D3_Narrow_Fast
  7: Both_D5_Narrow_Slow
  8: Both_D5_Narrow_Fast
  9: Both_D6_Narrow_Fast
  10: VibrotactileOnly_D7_Narrow_Slow
  11: Both_D6_Narrow_Slow
  12: VibrotactileOnly_D1_Narrow_Fast
  13: Both_D1_Narrow_Slow
  14: Both_D2_Narrow_Slow
  15: VibrotactileOnly_D1_Narrow_Slow
  16: Both_D7_Narrow_Fast
  17: VibrotactileOnly_D7_Narrow_Fast
  18: VibrotactileOnly_D2_Narrow_Slow
  19: VibrotactileOnly_D4_Narrow_Fast
  20: VibrotactileOnly_D5_Narrow_Slow
  21: VibrotactileOnly_D6_Narrow_Fast
  22: Both_D7_Narrow_Slow
  23: VibrotactileOnly_D6_Narrow_Slow
  24: V

### Wavelet computation

In [6]:
freqs = np.arange(4, 40, 1)

tfr_single_raw, tfr_average_raw = compute_tfr_by_event(
    interpolated_epochs_raw,
    freqs=freqs,
    baseline=(baseline_time_start, baseline_time_stop),
    baseline_mode="logratio"
)

tfr_single_merged, tfr_average_merged = compute_tfr_by_event(
    interpolated_epochs_merged,
    freqs=freqs,
    baseline=(baseline_time_start, baseline_time_stop),
    baseline_mode="logratio"
)

Computing TFR: VibrotactileOnly_D4_Narrow_Slow
Applying baseline correction (mode: logratio)
  Trials: 13
Computing TFR: VibrotactileOnly_D3_Narrow_Fast
Applying baseline correction (mode: logratio)
  Trials: 12
Computing TFR: VisualOnly_Narrow_Slow
Applying baseline correction (mode: logratio)
  Trials: 19
Computing TFR: VisualOnly_Narrow_Fast
Applying baseline correction (mode: logratio)
  Trials: 21
Computing TFR: Both_D4_Narrow_Slow
Applying baseline correction (mode: logratio)
  Trials: 18
Computing TFR: Both_D3_Narrow_Fast
Applying baseline correction (mode: logratio)
  Trials: 18
Computing TFR: Both_D5_Narrow_Slow
Applying baseline correction (mode: logratio)
  Trials: 16
Computing TFR: Both_D5_Narrow_Fast
Applying baseline correction (mode: logratio)
  Trials: 18
Computing TFR: Both_D6_Narrow_Fast
Applying baseline correction (mode: logratio)
  Trials: 14
Computing TFR: VibrotactileOnly_D7_Narrow_Slow
Applying baseline correction (mode: logratio)
  Trials: 3
Computing TFR: Both

### Save Time-Freq data

In [7]:
file_path_raw = os.path.join(data_path, subjectID, task, "Output", "TFR_raw")
filename = f"{subjectID}_{date}_{task}_raw"
save_paths = save_tfr_results(
    file_path=file_path_raw,
    file_name=filename,
    tfr_single_dict=tfr_single_raw,
    tfr_average_dict=tfr_average_raw
)

file_path_merged = os.path.join(data_path, subjectID, task, "Output", "TFR_merged")
filename = f"{subjectID}_{date}_{task}_merged"
save_paths = save_tfr_results(
    file_path=file_path_merged,
    file_name=filename,
    tfr_single_dict=tfr_single_raw,
    tfr_average_dict=tfr_average_raw
)


Saving single-trial TFRs...
  Saved VibrotactileOnly_D4_Narrow_Slow
  Saved VibrotactileOnly_D3_Narrow_Fast
  Saved VisualOnly_Narrow_Slow
  Saved VisualOnly_Narrow_Fast
  Saved Both_D4_Narrow_Slow
  Saved Both_D3_Narrow_Fast
  Saved Both_D5_Narrow_Slow
  Saved Both_D5_Narrow_Fast
  Saved Both_D6_Narrow_Fast
  Saved VibrotactileOnly_D7_Narrow_Slow
  Saved Both_D6_Narrow_Slow
  Saved VibrotactileOnly_D1_Narrow_Fast
  Saved Both_D1_Narrow_Slow
  Saved Both_D2_Narrow_Slow
  Saved VibrotactileOnly_D1_Narrow_Slow
  Saved Both_D7_Narrow_Fast
  Saved VibrotactileOnly_D7_Narrow_Fast
  Saved VibrotactileOnly_D2_Narrow_Slow
  Saved VibrotactileOnly_D4_Narrow_Fast
  Saved VibrotactileOnly_D5_Narrow_Slow
  Saved VibrotactileOnly_D6_Narrow_Fast
  Saved Both_D7_Narrow_Slow
  Saved VibrotactileOnly_D6_Narrow_Slow
  Saved VibrotactileOnly_D2_Narrow_Fast
  Saved Both_D2_Narrow_Fast
  Saved Both_D3_Narrow_Slow
  Saved VibrotactileOnly_D3_Narrow_Slow
  Saved Both_D4_Narrow_Fast
  Saved VibrotactileOnly_